<a href="https://colab.research.google.com/github/Podrimate/THz_sim_application/blob/main/notebooks/THz_User_Visual_Guide.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# THz User Visual Guide

This notebook explains what the main THzSim2 study and fit plots mean.

It is meant as a visual guide, not as the main production workflow. Use it when you want to understand:
- raw vs aligned vs processed traces
- what a good time-domain fit looks like
- what the residual means
- how to read the study heatmaps


## 0. Install And Import

Run the next cell first. In Colab it forces a fresh install from GitHub so notebook imports stay in sync with the latest repo version.


In [ ]:
import importlib
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
PACKAGE_IMPORT_OK = False
IMPORT_EXCEPTION = None
DEFAULT_PIP_SPEC = 'https://github.com/Podrimate/THz_sim_application/archive/refs/heads/main.zip'

def try_import_runtime_dependencies():
    global IMPORT_EXCEPTION
    try:
        import numpy  # noqa: F401
        import scipy  # noqa: F401
        import matplotlib  # noqa: F401
        return True
    except Exception as exc:
        IMPORT_EXCEPTION = exc
        return False

def try_import_thzsim2():
    global IMPORT_EXCEPTION
    try:
        importlib.invalidate_caches()
        import thzsim2  # noqa: F401
        return thzsim2
    except Exception as exc:
        IMPORT_EXCEPTION = exc
        return None

if IN_COLAB:
    pip_spec = os.environ.get('THZSIM_PIP_SPEC', DEFAULT_PIP_SPEC).strip()
    subprocess.call([sys.executable, '-m', 'pip', 'uninstall', '-y', 'thzsim2'])
    subprocess.check_call([
        sys.executable,
        '-m',
        'pip',
        'install',
        '-q',
        '--upgrade',
        '--force-reinstall',
        '--no-cache-dir',
        '--no-deps',
        pip_spec,
    ])
    sys.modules.pop('thzsim2', None)
    sys.modules.pop('thzsim2.notebook_api', None)

if not try_import_runtime_dependencies():
    raise RuntimeError(
        'The Python runtime dependencies are currently broken. '
        f'Last import error: {IMPORT_EXCEPTION!r}. '
        'In Colab use Runtime -> Factory reset runtime, then rerun the first cell. '
        "The notebook now installs only thzsim2 and leaves Colab's NumPy/SciPy/Matplotlib in place."
    )

thzsim2 = try_import_thzsim2()
PACKAGE_IMPORT_OK = thzsim2 is not None

if not PACKAGE_IMPORT_OK:
    for candidate in [Path.cwd(), Path.cwd().parent]:
        if (candidate / 'thzsim2').exists():
            sys.path.insert(0, str(candidate))
            thzsim2 = try_import_thzsim2()
            if thzsim2 is not None:
                PACKAGE_IMPORT_OK = True
                break

if not PACKAGE_IMPORT_OK:
    raise RuntimeError(
        'Could not import thzsim2. '
        f'Last import error: {IMPORT_EXCEPTION!r}. '
        'In Colab use Runtime -> Factory reset runtime and rerun the first cell; '
        'locally open the notebook inside the repo.'
    )

import matplotlib
import numpy
import scipy

print(f'Using thzsim2 from: {thzsim2.__file__}')
print(f'thzsim2 version: {getattr(thzsim2, "__version__", "unknown")}')
print(f'numpy version: {numpy.__version__}')
print(f'scipy version: {scipy.__version__}')
print(f'matplotlib version: {matplotlib.__version__}')


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display

from thzsim2.io.trace_csv import write_trace_csv
from thzsim2.notebook_api import (
    ConstantNK,
    Drude,
    Fit,
    Layer,
    Measurement,
    build_sample,
    build_fit_setup,
    build_study_setup,
    generate_reference_pulse,
    load_reference_csv,
    plot_trace_pair_preview,
    prepare_reference,
    prepare_trace_pair_for_fit,
    run_measured_fit_from_setup_json,
    run_study_from_setup_json,
    show_study_heatmaps,
    summarize_prepared_trace_pair,
    write_fit_setup_json,
    write_study_setup_json,
)

guide_root = Path.cwd() / 'thz_visual_guide_outputs'
guide_root.mkdir(parents=True, exist_ok=True)

def find_repo_path(relative_path):
    for candidate in [Path.cwd(), Path.cwd().parent]:
        candidate_path = candidate / relative_path
        if candidate_path.exists():
            return candidate_path.resolve()
    return None

LOCAL_TEST_DATA = find_repo_path('Test_data_for_fitter')
use_local_example = LOCAL_TEST_DATA is not None and (LOCAL_TEST_DATA / 'A11008858_transmission' / 'REFERENCE.csv').exists()

def note(text):
    display(Markdown(text))


## 1. Raw, Aligned, And Processed Traces

What you should look for:
- the raw traces are the imported measurements
- the aligned traces share the same sampled time axis, but the physical delay between reference and sample is preserved
- the processed window should still contain the strongest pulse in both traces


In [ ]:
from thzsim2.core.forward import simulate_sample_from_reference

if use_local_example:
    reference_csv = LOCAL_TEST_DATA / 'A11008858_transmission' / 'REFERENCE.csv'
    sample_csv = LOCAL_TEST_DATA / 'A11008858_transmission' / 'SAMPLE.csv'
    prepared_traces = prepare_trace_pair_for_fit(
        reference_csv,
        sample_csv,
        baseline_mode='auto_pre_pulse',
        baseline_window_samples=40,
        crop_mode='auto',
    )
    note('Using the local `Test_data_for_fitter/A11008858_transmission` example.')
else:
    synthetic_reference_input = generate_reference_pulse(
        model='sech_carrier',
        sample_count=512,
        dt_ps=0.03,
        time_center_ps=8.0,
        pulse_center_ps=5.0,
        tau_ps=0.22,
        f0_thz=0.9,
        amp=1.0,
        phi_rad=0.0,
    )
    synthetic_reference = prepare_reference(
        synthetic_reference_input,
        output_root=guide_root / 'synthetic_reference_runs',
        run_label='visual-guide-reference',
    )
    synthetic_sample = build_sample(
        layers=[
            Layer(name='film', thickness_um=120.0, material=ConstantNK(n=2.1, k=0.03)),
        ],
        reference=synthetic_reference,
        out_dir=synthetic_reference.run_dir / 'sample',
    )
    synthetic_simulation = simulate_sample_from_reference(
        synthetic_reference,
        synthetic_sample.resolved_stack,
        measurement=Measurement(mode='transmission', angle_deg=18.0, polarization='mixed', polarization_mix=0.5),
    )
    reference_csv = guide_root / 'synthetic_reference.csv'
    sample_csv = guide_root / 'synthetic_sample.csv'
    write_trace_csv(reference_csv, synthetic_reference.trace)
    write_trace_csv(sample_csv, synthetic_simulation['trace_data'])
    prepared_traces = prepare_trace_pair_for_fit(
        synthetic_reference.trace,
        synthetic_simulation['trace_data'],
        baseline_mode='auto_pre_pulse',
        baseline_window_samples=40,
        crop_mode='auto',
    )
    note('Local example data was not found, so this guide is using a synthetic transmission example instead.')

summary = summarize_prepared_trace_pair(prepared_traces)
summary


In [ ]:
plot_trace_pair_preview(prepared_traces)


Interpretation:
- if the green processed window cuts off the dominant pulse, the fit setup is wrong
- if the baseline estimate is reasonable, the processed traces should oscillate around zero away from the pulse
- the sample pulse may arrive later than the reference pulse; this delay is physical and should not be removed


## 2. Measured Time-Domain Fit

The next cells run a small measured fit using the same local example data.


In [ ]:
fit_setup = build_fit_setup(
    reference_trace={'path': reference_csv},
    sample_trace={'path': sample_csv},
    preprocessing={
        'baseline_mode': 'auto_pre_pulse',
        'baseline_window_samples': 40,
        'crop_mode': 'auto',
    },
    layers=[
        Layer(
            name='drude_layer',
            thickness_um=Fit(550.0, abs_min=300.0, abs_max=800.0, label='film_thickness_um'),
            material=Drude(
                eps_inf=12.0,
                plasma_freq_thz=Fit(1.1, abs_min=0.1, abs_max=3.0, label='film_plasma_freq_thz'),
                gamma_thz=Fit(0.08, abs_min=0.005, abs_max=0.5, label='film_gamma_thz'),
            ),
        )
    ],
    measurement=Measurement(
        mode='transmission',
        angle_deg=Fit(8.0, abs_min=0.0, abs_max=25.0, label='measurement_angle_deg'),
        polarization='mixed',
        polarization_mix=Fit(0.5, abs_min=0.0, abs_max=1.0, label='measurement_polarization_mix'),
    ),
    optimizer={
        'method': 'L-BFGS-B',
        'options': {'maxiter': 8},
        'global_options': {'maxiter': 1, 'popsize': 5, 'seed': 11},
        'fd_rel_step': 1e-4,
    },
    max_internal_reflections=2,
    out_dir=guide_root / 'fit_run',
)
fit_setup_path = write_fit_setup_json(guide_root / 'fit_setup.json', fit_setup)
measured_fit_result = run_measured_fit_from_setup_json(fit_setup_path)
measured_fit_result.fit_result['recovered_parameters']


In [ ]:
fit_trace = np.asarray(measured_fit_result.fit_result['fitted_simulation']['sample_trace'], dtype=float)
processed_sample = measured_fit_result.prepared_traces.processed_sample
processed_reference = measured_fit_result.prepared_traces.processed_reference
residual = np.asarray(measured_fit_result.fit_result['residual_trace'], dtype=float)
omega = np.asarray(measured_fit_result.fit_result['fitted_simulation']['omega_rad_s'], dtype=float)
freq_thz = omega / (2.0 * np.pi * 1e12)
transfer = np.asarray(measured_fit_result.fit_result['fitted_simulation']['transfer_function'])

fig, axes = plt.subplots(3, 1, figsize=(10, 11), sharex=False)
axes[0].plot(processed_reference.time_ps, processed_reference.trace, label='processed reference', alpha=0.8)
axes[0].plot(processed_sample.time_ps, processed_sample.trace, label='processed sample', linewidth=1.4)
axes[0].plot(processed_sample.time_ps, fit_trace, label='fit', linewidth=1.4)
axes[0].set_title('Measured Time-Domain Fit')
axes[0].set_xlabel('Time (ps)')
axes[0].set_ylabel('Signal')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(processed_sample.time_ps, residual, label='residual')
axes[1].set_title('Residual Trace')
axes[1].set_xlabel('Time (ps)')
axes[1].set_ylabel('Residual')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

axes[2].plot(freq_thz, np.abs(transfer), label='|sample response|')
axes[2].set_title('Optical Response Magnitude')
axes[2].set_xlabel('Frequency (THz)')
axes[2].set_ylabel('Magnitude')
axes[2].grid(True, alpha=0.3)
axes[2].legend()
fig.tight_layout()
plt.show()


Interpretation:
- the fit should track the processed sample, especially around the main pulse and the strongest ringing
- the residual should be centered around zero and noticeably smaller than the fitted signal
- the optical response magnitude should look smooth and physically plausible across frequency


## 3. Study Heatmaps

The next cells run a small synthetic study and show the heatmaps.


In [ ]:
study_setup = build_study_setup(
    reference={
        'kind': 'generated_pulse',
        'generate': {
            'model': 'sech_carrier',
            'sample_count': 256,
            'dt_ps': 0.03,
            'time_center_ps': 8.0,
            'pulse_center_ps': 5.0,
            'tau_ps': 0.22,
            'f0_thz': 0.9,
            'amp': 1.0,
            'phi_rad': 0.0,
        },
        'prepare': {
            'output_root': guide_root / 'study_runs',
            'run_label': 'visual-guide-study',
        },
    },
    layers=[
        Layer(
            name='film',
            thickness_um=Fit(150.0, abs_min=90.0, abs_max=220.0, label='film_thickness_um'),
            material=ConstantNK(
                n=Fit(2.1, abs_min=1.7, abs_max=2.5, label='film_n'),
                k=0.03,
            ),
        )
    ],
    measurement={
        'mode': 'transmission',
        'angle_deg': 25.0,
        'polarization': 'mixed',
        'polarization_mix': 0.5,
        'reference_standard': {'kind': 'identity'},
    },
    study={
        'truth': {
            'layers[0].thickness_um': [120.0, 150.0, 180.0],
            'layers[0].material.n': [1.9, 2.1, 2.3],
        },
        'noise_dynamic_range_db': 85.0,
        'replicates': 1,
        'seed': 9,
        'optimizer': {
            'method': 'L-BFGS-B',
            'options': {'maxiter': 8},
            'global_options': {'maxiter': 1, 'popsize': 4, 'seed': 9},
            'fd_rel_step': 1e-4,
        },
        'out_dir': guide_root / 'study_run',
    },
)
study_setup_path = write_study_setup_json(guide_root / 'study_setup.json', study_setup)
study_result = run_study_from_setup_json(study_setup_path)
study_result.summary_rows[:3]


In [ ]:
show_study_heatmaps(study_result, contains='normalized-mse__', max_images=6)
show_study_heatmaps(study_result, contains='relative-l2__', max_images=6)
show_study_heatmaps(study_result, contains='signed-err', max_images=6)


Interpretation:
- `normalized_mse` heatmaps show where the time-domain mismatch is strongest after amplitude scaling
- `relative_l2` heatmaps show overall waveform mismatch in a normalized energy sense
- `signed-err__...` heatmaps show parameter bias: positive means `true - fit` is positive, negative means the fit overshoots the true value


## 4. What To Send Back For Checking

For fitting problems, the best review file is `fit_setup.json`.

For study problems, the best review file is `study_setup.json`.

These files preserve the exact settings needed to rerun and inspect the result.
